<div style="text-align: center;">
  <span style="
      color: lightskyblue;
      font-family: 'Segoe UI', 'Noto Sans KR', 'Malgun Gothic', sans-serif;
      font-weight: 700;
      font-size: 48px;
      letter-spacing: 0.5px;
      line-height: 1.2;
  ">
    Demo File for Pick and Place
  </span>
</div>

<div style="color:white; font-size:18px; font-weight:400; line-height:1.5;">
This is the Demo File for 260723<br><br>
Scenario:<br>
&nbsp;&nbsp;&nbsp;&nbsp;1. In PML Lab, the robot turns left 90 degrees and takes the leftmost black block with its left arm.<br>
&nbsp;&nbsp;&nbsp;&nbsp;2. The robot turns right 90 degrees and transfers the object from the left arm to the right arm.<br>
&nbsp;&nbsp;&nbsp;&nbsp;3. The robot places the object into the yellow box using its right arm.<br><br>
If you want to use ALL cases (left → left, right → right, left → right, right → left),<br>
use <code>demoFile_pickAndPlace_ALLcases.ipynb</code>.
</div>

<span style="
    color:#B8860B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:50px;
    letter-spacing:1px;
    line-height:1.2;
">
Import Library
</span>

In [2]:
import rby1_sdk as rby
import numpy as np
import logging
import time
import pyrealsense2 as rs
import cv2

import sys
from pathlib import Path
# Path.cwd()           : folder "260723", 
# Path.cwd().parents[0]: folder "Demonstration", 
# Path.cwd().parents[1]: folder "RBY1_library_ysw"
sys.path.append(str(Path.cwd().parents[1]))                  
import rby1_lib_ysw as ysw

np.set_printoptions(suppress=True, precision=4, linewidth=300)  

# In .ipynb, show logging 
logging.basicConfig(
    level=logging.INFO,
    format="[%(levelname)s] %(message)s",
    # format="%(asctime)s [%(levelname)s] %(message)s",
    stream=sys.stdout,
    force=True,
)

<span style="
    color:#B8860B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:50px;
    letter-spacing:1px;
    line-height:1.2;
">
Initialization [Connect + PowerON]
</span>

In [2]:
# robotConfig = ysw.setRobotConfig(ysw.MODE_SIM, ysw.MODEL_A)                 # For Simulation
# robotConfig = ysw.setRobotConfig(ysw.MODE_LOCAL_COMPUTER, ysw.MODEL_A)      # For Real Robot + Laptop (RBY1-Hotspot)
robotConfig = ysw.setRobotConfig(ysw.MODE_DIRECT_UPC, ysw.MODEL_A)          # For Real Robot + UPC

robot = ysw.initialize(robotConfig.mode, robotConfig.address, robotConfig.model, option=ysw.CONNECT_AND_POWER)
# robot = ysw.initialize(robotConfig.mode, robotConfig.address, robotConfig.model, option=ysw.CONNECT_ONLY)
if (robot != None):
    model = robot.model()

*************************** [Initialization Starts] ****************************

[INFO] SUCCESS: Connected to the Robot!
[INFO] SUCCESS: Power ON the Robot!
[INFO] SUCCESS: Servo ON the Robot!
[INFO] SUCCESS: Enable the Control Manager!
[INFO] SUCCESS: Enable the Grippers!

*************************** [Intialization is Done] ****************************





<span style="
    color:#B8860B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:50px;
    letter-spacing:1px;
    line-height:1.2;
">
Power OFF
</span>

In [7]:
ysw.powerOFF(robot, robotConfig)

****************************** [powerOFF Starts] *******************************

[INFO] SUCCESS: Disable Grippers!
[INFO] SUCCESS: Disable the Control Manager!

****************************** [PowerOFF is Done] ******************************





True

<span style="
    color:#B8860B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:50px;
    letter-spacing:1px;
    line-height:1.2;
">
Posture [Initial]
</span>

In [3]:
desired_pos = ysw.DEMO_BASIC_POS_260723     # ZERO_POS, READY_POS, PACK_POS, DEMO_BASIC_POS_260723
ysw.positionCtrl_joint_22dof(robot, desired_pos, minimum_time=10.0, control_hold_time=0.5)

******************** [22-DoF Joint Position Control Starts] ********************


********************* [22-DoF Joint Position Control Ends] *********************





True

<span style="
    color:#B8860B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:50px;
    letter-spacing:1px;
    line-height:1.2;
">
Posture [Turn to Shelf]
</span>

In [4]:
desired_pos = ysw.DEMO_TURN_POS_260723     # ZERO_POS, READY_POS, PACK_POS, DEMO_TURN_POS_260723
ysw.positionCtrl_joint_22dof(robot, desired_pos, minimum_time=10.0, control_hold_time=0.5)

******************** [22-DoF Joint Position Control Starts] ********************


********************* [22-DoF Joint Position Control Ends] *********************





True

<span style="
    color:#B8860B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:50px;
    letter-spacing:1px;
    line-height:1.2;
">
Leftmost Object Detection during 4 seconds
</span>

In [7]:
d435i = ysw.D435i(rgb_on=True, depth_on=True, pixel=(640, 480), fps=30)
d435i.initialize()

record_duration = 4.0  # [sec]

leftmost_object_samples = []        # 제일 왼쪽 물체

d435i.startPipeline()

try:
    start_time = time.time()

    while True:
        elapsed_time = time.time() - start_time

        if elapsed_time >= record_duration:
            break

        color_frame, depth_frame = d435i.getRawFrames()

        if color_frame is None or depth_frame is None:
            continue

        color_image, depth_image = d435i.frameToImage(color_frame, depth_frame)

        detected_objects, color_image, mask = d435i.detectColorObjectContour3D(
            color_image=color_image,
            depth_frame=depth_frame,
            mask_list=ysw.HSV_MASK_BLACK,
            minimum_area=1000,       
            maximum_depth=0.8,                   # ignore distant object larger than x[m] 
        )

        if len(detected_objects) == 0:
            continue

        # 카메라 기준 X 좌표가 작을수록 이미지 왼쪽
        # RealSense/D435i 기준: X+ = image right
        obj_list_from_left = sorted(
            detected_objects,
            key=lambda obj: obj["center_3d"][0],
            reverse=False
        )

        # 제일 왼쪽 물체 하나만 선택
        obj = obj_list_from_left[0]

        # 제일 왼쪽 물체 sample만 저장
        leftmost_object_samples.append({
            "area_2d":      float(obj["area_2d"]),
            "pixel_center": np.array(obj["pixel_center"], dtype=np.float64),
            "center_3d":    np.array(obj["center_3d"], dtype=np.float64),
            "bbox_3d_size": np.array(obj["bbox_3d_size"], dtype=np.float64),
            "bbox_3d_min":  np.array(obj["bbox_3d_min"], dtype=np.float64),
            "bbox_3d_max":  np.array(obj["bbox_3d_max"], dtype=np.float64),
        })

finally:
    d435i.stopPipeline()
    d435i.destroyAllWindows()



# =========================
# Average leftmost object
# =========================

leftmost_obj_avg_sample = None

if len(leftmost_object_samples) > 0:
    leftmost_obj_avg_sample = ysw.average_detectColorObjectContour3D(
        leftmost_object_samples
    )

print("\n========== Detection Result ==========")
print("Only the LEFTMOST object is used\n")

if leftmost_obj_avg_sample is None:
    print("No object detected.")

else:
    print("\n[Leftmost Object]")
    print(f"num_samples             : {leftmost_obj_avg_sample['num_samples']}")
    print(f"area_2d_avg             : {leftmost_obj_avg_sample['area_2d_avg']:.1f}")
    print(f"area_2d_median          : {leftmost_obj_avg_sample['area_2d_median']:.1f}")
    print(f"center_3d_avg           : {np.round(leftmost_obj_avg_sample['center_3d_avg'], 4)} m")
    print(f"center_3d_median        : {np.round(leftmost_obj_avg_sample['center_3d_median'], 4)} m")
    print(f"bbox_3d_size_avg        : {np.round(leftmost_obj_avg_sample['bbox_3d_size_avg'], 4)} m")
    print(f"bbox_3d_size_median     : {np.round(leftmost_obj_avg_sample['bbox_3d_size_median'], 4)} m")

[INFO] D435i pipeline started

========== Detection Result ==========
Only the LEFTMOST object is used


[Leftmost Object]
num_samples             : 60
area_2d_avg             : 5320.3
area_2d_median          : 5318.0
center_3d_avg           : [-0.1796  0.1355  0.6489] m
center_3d_median        : [-0.1797  0.1353  0.649 ] m
bbox_3d_size_avg        : [0.0727 0.1605 0.2832] m
bbox_3d_size_median     : [0.0728 0.1605 0.285 ] m


<span style="
    color:#B8860B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:50px;
    letter-spacing:1px;
    line-height:1.2;
">
Calculate IK for Leftmost Object
</span>

In [8]:
# {A}: vision frame
# {B}: "link_head_2" frame
# {S}: "link_torso_5" frame 
# {M}: Goal frame (object) 
# {M1}: Goal frame (object) before open the gripper 
# {M2}: Goal frame (object) after open the gripper 
# {N}: Goal frame (collecting box) 
# {C}: ee_finger_l1/r1 frame
# {D}: final fingertip frame -> same orientation of {C}

manual_offset = 0.05  # [m] for forward x-direction relative to "link_torso_5"

leftmost_obj_pos    = leftmost_obj_avg_sample["center_3d_avg"]
leftmost_obj_3Dsize = leftmost_obj_avg_sample["bbox_3d_size_avg"]

P_BM = ysw.convertPos_vision_to_linkhead2(leftmost_obj_pos)

fk_result = ysw.get_FK(robot, "link_torso_5", "link_head_2")
T_SB = fk_result["T"]
R_SB = fk_result["rotation"]
P_SB = fk_result["position"]

P_BM_extended = np.append(P_BM, 1.0)
P_SM = T_SB @ P_BM_extended
P_SM = P_SM[:3]

moved_arm = None
if (P_BM[1] <= 0):     
    moved_arm = "right"
else:     
    moved_arm = "left"

if (moved_arm == "right"):
    R1 = ysw.rotMat_z(90.0*ysw.DEG2RAD)      # "link_torso_5" frame 기준으로 z축 +90도 회전
    R2 = ysw.rotMat_x(-90.0*ysw.DEG2RAD)     # After R1, converted "link_torso_5" frame 기준으로 x축 -90도 회전
    R_SC_desired = R1 @ R2                   # Make Grasping orientation same as D435i frame orientation angle (it does not mean that x,y,z axis directions are same)  
elif (moved_arm == "left"):
    R1 = ysw.rotMat_x(90.0*ysw.DEG2RAD)      # "link_torso_5" frame 기준으로 x축 +90도 회전
    R2 = ysw.rotMat_y(-90.0*ysw.DEG2RAD)     # After R1, converted "link_torso_5" frame 기준으로 y축 -90도 회전
    R_SC_desired = R1 @ R2                   # Make Grasping orientation same as D435i frame orientation angle (it does not mean that x,y,z axis directions are same) 

P_SM1 = P_SM + np.array([-abs(ysw.GRIPPER_OFFSET_BEFORE_GRASP), 0.0, 0.0])      # x-offset before open the gripper
R_SM1 = R_SC_desired
T_SM1 = ysw.make_HT(R_SM1, P_SM1)    

P_SM2 = P_SM + np.array([manual_offset, 0.0, 0.0])      # Theoretically manual_offset should be zero
R_SM2 = R_SC_desired
T_SM2 = ysw.make_HT(R_SM2, P_SM2)    

# "ee_finger_l1/r1" to final end-effector -> should be physically measured
pos_from_eeFinger_to_fingertip = np.array([0, 0, ysw.GRIPPER_Z_OFFSET_FROM_LINK_EE_FINGER]) # P_CD
P_CD = pos_from_eeFinger_to_fingertip
T_CD = ysw.make_HT(np.eye(3), P_CD)    

T_SC_target_before_open_gripper = T_SM1 @ np.linalg.inv(T_CD)
T_SC_target_after_open_gripper  = T_SM2 @ np.linalg.inv(T_CD)
print(f"Before open gripper: \n{T_SC_target_before_open_gripper}\n")
print(f"After open gripper: \n{T_SC_target_after_open_gripper}\n")
print(f"Moving Arm: {moved_arm}")


*************** [FK of 'link_head_2' from 'link_torso_5' Starts] ***************

[Result]

T:
[[ 0.9397  0.0005  0.342   0.022 ]
 [-0.0005  1.     -0.0002  0.    ]
 [-0.342   0.      0.9397  0.2001]
 [ 0.      0.      0.      1.    ]]


position [m]: [0.022  0.     0.2001]
rpy [rad]: [ 0.      0.3491 -0.0005]
rpy [deg]: [ 0.     20.0002 -0.0283]

**************** [FK of 'link_head_2' from 'link_torso_5' Ends] ****************



Before open gripper: 
[[ 0.      0.     -1.      0.5189]
 [-1.      0.     -0.      0.2018]
 [ 0.      1.      0.     -0.0834]
 [ 0.      0.      0.      1.    ]]

After open gripper: 
[[ 0.      0.     -1.      0.5989]
 [-1.      0.     -0.      0.2018]
 [ 0.      1.      0.     -0.0834]
 [ 0.      0.      0.      1.    ]]

Moving Arm: left


<span style="
    color:#B8860B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:50px;
    letter-spacing:1px;
    line-height:1.2;
">
Move to Leftmost object on Shelf (before open gripper)
</span>

In [9]:
pos_before_open_gripper = T_SC_target_before_open_gripper[:3,3]
ori_before_open_gripper = ysw.rotMat_to_rpy(T_SC_target_before_open_gripper[:3,:3])       

if (moved_arm == "right"):      # Object is in right side of robot
    cartesian_position_targets_sample = [    
                                            {
                                                "reference_link": "link_torso_5",
                                                "target_link": "ee_finger_r1",
                                                "pos": pos_before_open_gripper,                  
                                                "rpy": ori_before_open_gripper,                    
                                            },                 
                                        ]
    
elif (moved_arm == "left"):       # Object is in left side of robot
    cartesian_position_targets_sample = [    
                                            {
                                                "reference_link": "link_torso_5",
                                                "target_link": "ee_finger_l1",
                                                "pos": pos_before_open_gripper,                  
                                                "rpy": ori_before_open_gripper,                    
                                            },                 
                                        ]


# [rv = "function"] form is Necessary when "get_return=False"
rv = ysw.positionCtrl_cartesian_wholeBody(
    robot, 
    position_targets=cartesian_position_targets_sample, 
    joint_targets=None,
    minimum_time=10,
    control_hold_time=0.5,
    get_return=True
)

*************** [Cartesian Position Control(Whole-Body) Starts] ****************

[INFO] Add (Whole-Body) Cartesian target 0: 'link_torso_5'->'ee_finger_l1', pos=[ 0.5189  0.2018 -0.0834], rpy=[ 1.5708 -0.     -1.5708]

**************** [Cartesian Position Control(Whole-Body) Ends] *****************





<span style="
    color:#B8860B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:50px;
    letter-spacing:1px;
    line-height:1.2;
">
Open Gripper before Grasp
</span>

In [10]:
object_meanDim = np.mean(leftmost_obj_3Dsize)       # mean value of 3D size of object to grasp
desiredGripperPos = object_meanDim 

gripper = ysw.Gripper()
gripper.connect(device_name=ysw.GRIPPER_DEVICE_NAME, protocol_version=ysw.DXL_PROTOCOL_VERSION, baudrate=ysw.DXL_BAUDRATE)
gripper.test_ping()


# desiredGripperPos_mm = desiredGripperPos * 1000.0 + ysw.GRIPPER_SAFETY_OPEN_OFFSET
desiredGripperPos_mm = desiredGripperPos * 1000.0
if desiredGripperPos_mm >= ysw.GRIPPER_POS_RANGE_LINEAR:
    desiredGripperPos_mm = ysw.GRIPPER_POS_RANGE_LINEAR - 3.0

if (moved_arm == "right"):      # Object is in right side of robot
    gripper.initialize_setting(dxl_id=ysw.GRIPPER_ID_R, operation_mode=ysw.DXL_CURRENT_BASED_POSITION_CONTROL_MODE)
    gripper.current_based_position_control_linear(ysw.GRIPPER_ID_R, current_ampere_mag=0.02, linear_pos=desiredGripperPos_mm)
    time.sleep(1)
    gripper.disable(ysw.GRIPPER_ID_R)
elif (moved_arm == "left"):       # Object is in left side of robot
    gripper.initialize_setting(dxl_id=ysw.GRIPPER_ID_L, operation_mode=ysw.DXL_CURRENT_BASED_POSITION_CONTROL_MODE)
    gripper.current_based_position_control_linear(ysw.GRIPPER_ID_L, current_ampere_mag=0.02, linear_pos=desiredGripperPos_mm)
    time.sleep(1)
    gripper.disable(ysw.GRIPPER_ID_L)

************************* [Gripper Connection Starts] **************************

[INFO] SUCCESS: open the port: /dev/rby1_gripper
[INFO] SUCCESS: change the baudrate 2000000

************************* [Gripper Connection is Done] *************************



************************** [Gripper Ping Test Starts] **************************

[INFO] [DXL ID:0] SUCCESS: Dynamixel model number is 1030
[INFO] [DXL ID:1] SUCCESS: Dynamixel model number is 1030

************************* [Gripper Ping Test is Done] **************************



********************* [Gripper [DXL ID:1] setting Starts] **********************

[INFO] SUCCESS: [DXL ID:1] Torque is Disabled
[INFO] SUCCESS: [DXL ID:1] Set Current-based Position Control Mode
[INFO] SUCCESS: [DXL ID:1] Torque is Enabled

********************* [Gripper [DXL ID:1] setting is Done] *********************



********** [Gripper [DXL ID:1] Current-based Position Control Starts] **********

Desired Linear Position = 88.0mm
[INFO] [DXL ID:1]

<span style="
    color:#B8860B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:50px;
    letter-spacing:1px;
    line-height:1.2;
">
Move to Leftmost object on Shelf (after open gripper)
</span>

In [11]:
pos_after_open_gripper = T_SC_target_after_open_gripper[:3,3]
ori_after_open_gripper = ysw.rotMat_to_rpy(T_SC_target_after_open_gripper[:3,:3])       

if (moved_arm == "right"):      # Object is in right side of robot
    cartesian_position_targets_sample = [    
                                            {
                                                "reference_link": "link_torso_5",
                                                "target_link": "ee_finger_r1",
                                                "pos": pos_after_open_gripper,                  
                                                "rpy": ori_after_open_gripper,                    
                                            },                 
                                        ]
    
elif (moved_arm == "left"):       # Object is in left side of robot
    cartesian_position_targets_sample = [    
                                            {
                                                "reference_link": "link_torso_5",
                                                "target_link": "ee_finger_l1",
                                                "pos": pos_after_open_gripper,                  
                                                "rpy": ori_after_open_gripper,                    
                                            },                 
                                        ]


# [rv = "function"] form is Necessary when "get_return=False"
rv = ysw.positionCtrl_cartesian_wholeBody(
    robot, 
    position_targets=cartesian_position_targets_sample, 
    joint_targets=None,
    minimum_time=10,
    control_hold_time=0.5,
    get_return=True
)

*************** [Cartesian Position Control(Whole-Body) Starts] ****************

[INFO] Add (Whole-Body) Cartesian target 0: 'link_torso_5'->'ee_finger_l1', pos=[ 0.5989  0.2018 -0.0834], rpy=[ 1.5708 -0.     -1.5708]

**************** [Cartesian Position Control(Whole-Body) Ends] *****************





<span style="
    color:#B8860B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:50px;
    letter-spacing:1px;
    line-height:1.2;
">
Close Gripper to Grasp
</span>

In [12]:
grasp_current = 0.2     # It can be changed

gripper = ysw.Gripper()
gripper.connect(device_name=ysw.GRIPPER_DEVICE_NAME, protocol_version=ysw.DXL_PROTOCOL_VERSION, baudrate=ysw.DXL_BAUDRATE)
gripper.test_ping()

if (moved_arm=="right"):      # Object is in right side of robot
    gripper.initialize_setting(dxl_id=ysw.GRIPPER_ID_R, operation_mode=ysw.DXL_CURRENT_CONTROL_MODE)
    gripper.enable(ysw.GRIPPER_ID_R)
    gripper.write_current_ampere(ysw.GRIPPER_ID_R, grasp_current)
elif (moved_arm=="left"):       # Object is in left side of robot
    gripper.initialize_setting(dxl_id=ysw.GRIPPER_ID_L, operation_mode=ysw.DXL_CURRENT_CONTROL_MODE)
    gripper.enable(ysw.GRIPPER_ID_L)
    gripper.write_current_ampere(ysw.GRIPPER_ID_L, grasp_current)

************************* [Gripper Connection Starts] **************************

[INFO] SUCCESS: open the port: /dev/rby1_gripper
[INFO] SUCCESS: change the baudrate 2000000

************************* [Gripper Connection is Done] *************************



************************** [Gripper Ping Test Starts] **************************

[INFO] [DXL ID:0] SUCCESS: Dynamixel model number is 1030
[INFO] [DXL ID:1] SUCCESS: Dynamixel model number is 1030

************************* [Gripper Ping Test is Done] **************************



********************* [Gripper [DXL ID:1] setting Starts] **********************

[INFO] SUCCESS: [DXL ID:1] Torque is Disabled
[INFO] SUCCESS: [DXL ID:1] Set Current Control Mode
[INFO] SUCCESS: [DXL ID:1] Torque is Enabled

********************* [Gripper [DXL ID:1] setting is Done] *********************



********************** [Gripper [DXL ID:1] Enable Starts] **********************

[INFO] SUCCESS: [DXL ID:1] Torque is Enabled

*******************

<span style="
    color:#B8860B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:50px;
    letter-spacing:1px;
    line-height:1.2;
">
Slightly float the object (to avoid collision)
</span>

In [13]:
if (moved_arm == "right"):      # Object is in right side of robot
    cartesian_position_targets_sample = [    
                                            {
                                                "reference_link": "link_torso_5",
                                                "target_link": "ee_finger_r1",
                                                "pos": pos_after_open_gripper + np.array([-0.05, 0.0, 0.05]),                  
                                                "rpy": ori_after_open_gripper,                    
                                            },                 
                                        ]
    
elif (moved_arm == "left"):       # Object is in left side of robot
    cartesian_position_targets_sample = [    
                                            {
                                                "reference_link": "link_torso_5",
                                                "target_link": "ee_finger_l1",
                                                "pos": pos_after_open_gripper + np.array([-0.05, 0.0, 0.05]),                  
                                                "rpy": ori_after_open_gripper,                    
                                            },                 
                                        ]


# [rv = "function"] form is Necessary when "get_return=False"
rv = ysw.positionCtrl_cartesian_wholeBody(
    robot, 
    position_targets=cartesian_position_targets_sample, 
    joint_targets=None,
    minimum_time=10,
    control_hold_time=0.5,
    get_return=True
)

*************** [Cartesian Position Control(Whole-Body) Starts] ****************

[INFO] Add (Whole-Body) Cartesian target 0: 'link_torso_5'->'ee_finger_l1', pos=[ 0.5489  0.2018 -0.0334], rpy=[ 1.5708 -0.     -1.5708]

**************** [Cartesian Position Control(Whole-Body) Ends] *****************





<span style="
    color:#B8860B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:50px;
    letter-spacing:1px;
    line-height:1.2;
">
Move to Table
</span>

In [14]:
desired_pos = ysw.DEMO_AFTER_GRASP_POS_260723 
ysw.positionCtrl_joint_22dof(robot, desired_pos, minimum_time=10.0, control_hold_time=0.5)

******************** [22-DoF Joint Position Control Starts] ********************


********************* [22-DoF Joint Position Control Ends] *********************





True

In [15]:
desired_pos = ysw.DEMO_BASIC_POS_260723 
ysw.positionCtrl_joint_22dof(robot, desired_pos, minimum_time=10.0, control_hold_time=0.5)

******************** [22-DoF Joint Position Control Starts] ********************


********************* [22-DoF Joint Position Control Ends] *********************





True

In [16]:
desired_pos = ysw.DEMO_VISION_BASIC_POS_260723 
ysw.positionCtrl_joint_22dof(robot, desired_pos, minimum_time=10.0, control_hold_time=0.5)

******************** [22-DoF Joint Position Control Starts] ********************


********************* [22-DoF Joint Position Control Ends] *********************





True

<span style="
    color:#B8860B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:50px;
    letter-spacing:1px;
    line-height:1.2;
">
Yellow Collecting Box Detection during 4 seconds
</span>

In [17]:
d435i = ysw.D435i(rgb_on=True, depth_on=True, pixel=(640, 480), fps=30)
d435i.initialize()

record_duration = 4.0  # [sec]

box_samples = []        # box

d435i.startPipeline()

try:
    start_time = time.time()

    while True:
        elapsed_time = time.time() - start_time

        if elapsed_time >= record_duration:
            break

        color_frame, depth_frame = d435i.getRawFrames()

        if color_frame is None or depth_frame is None:
            continue

        color_image, depth_image = d435i.frameToImage(color_frame, depth_frame)

        detected_objects, color_image, mask = d435i.detectColorObjectContour3D(
            color_image=color_image,
            depth_frame=depth_frame,
            mask_list=ysw.HSV_MASK_YELLOW,
            minimum_area=10000,       
            maximum_depth=1.5,                   # ignore distant object larger than x[m] 
        )

        if len(detected_objects) == 0:
            continue

        # The largest yellow object = box
        largest_obj = max(
            detected_objects,
            key=lambda obj: obj["area_2d"]
        )

        # [Largest area] sample
        box_samples.append({
            "area_2d":      float(largest_obj["area_2d"]),
            "pixel_center": np.array(largest_obj["pixel_center"], dtype=np.float64),
            "center_3d":    np.array(largest_obj["center_3d"], dtype=np.float64),
            "bbox_3d_size": np.array(largest_obj["bbox_3d_size"], dtype=np.float64),
            "bbox_3d_min":  np.array(largest_obj["bbox_3d_min"], dtype=np.float64),
            "bbox_3d_max":  np.array(largest_obj["bbox_3d_max"], dtype=np.float64),
        })

finally:
    d435i.stopPipeline()
    d435i.destroyAllWindows()



# =========================
# Average leftmost object
# =========================

box_obj_avg_sample = None

if len(box_samples) > 0:
    box_obj_avg_sample = ysw.average_detectColorObjectContour3D(
        box_samples
    )

print("\n========== Detection Result ==========")
print("Only the Largest Box is used\n")

if box_obj_avg_sample is None:
    print("No object detected.")

else:
    print("\n[Largest Box Object]")
    print(f"num_samples             : {box_obj_avg_sample['num_samples']}")
    print(f"area_2d_avg             : {box_obj_avg_sample['area_2d_avg']:.1f}")
    print(f"area_2d_median          : {box_obj_avg_sample['area_2d_median']:.1f}")
    print(f"center_3d_avg           : {np.round(box_obj_avg_sample['center_3d_avg'], 4)} m")
    print(f"center_3d_median        : {np.round(box_obj_avg_sample['center_3d_median'], 4)} m")
    print(f"bbox_3d_size_avg        : {np.round(box_obj_avg_sample['bbox_3d_size_avg'], 4)} m")
    print(f"bbox_3d_size_median     : {np.round(box_obj_avg_sample['bbox_3d_size_median'], 4)} m")

[INFO] D435i pipeline started

========== Detection Result ==========
Only the Largest Box is used


[Largest Box Object]
num_samples             : 21
area_2d_avg             : 48582.5
area_2d_median          : 48518.5
center_3d_avg           : [0.2159 0.1126 0.7392] m
center_3d_median        : [0.2191 0.1088 0.783 ] m
bbox_3d_size_avg        : [0.328  0.3366 0.5899] m
bbox_3d_size_median     : [0.3256 0.3508 0.551 ] m


<span style="
    color:#B8860B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:50px;
    letter-spacing:1px;
    line-height:1.2;
">
Move to pre-[Left to Right] posture
</span>

In [18]:
desired_pos = ysw.DEMO_LEFT_TO_RIGHT_POS_BEFORE_260723  
ysw.positionCtrl_joint_22dof(robot, desired_pos, minimum_time=10.0, control_hold_time=0.5)

******************** [22-DoF Joint Position Control Starts] ********************


********************* [22-DoF Joint Position Control Ends] *********************





True

<span style="
    color:#B8860B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:50px;
    letter-spacing:1px;
    line-height:1.2;
">
Open Gripper before [Left to Right gripper]
</span>

In [19]:
object_meanDim = np.mean(leftmost_obj_3Dsize)       # mean value of 3D size of object to grasp
desiredGripperPos = object_meanDim 

gripper = ysw.Gripper()
gripper.connect(device_name=ysw.GRIPPER_DEVICE_NAME, protocol_version=ysw.DXL_PROTOCOL_VERSION, baudrate=ysw.DXL_BAUDRATE)
gripper.test_ping()

# desiredGripperPos_mm = desiredGripperPos * 1000.0 + ysw.GRIPPER_SAFETY_OPEN_OFFSET
desiredGripperPos_mm = desiredGripperPos * 1000.0
if desiredGripperPos_mm >= ysw.GRIPPER_POS_RANGE_LINEAR:
    desiredGripperPos_mm = ysw.GRIPPER_POS_RANGE_LINEAR - 3.0

if (moved_arm == "left"):       # Object is in left side of robot
    gripper.initialize_setting(dxl_id=ysw.GRIPPER_ID_R, operation_mode=ysw.DXL_CURRENT_BASED_POSITION_CONTROL_MODE)
    gripper.current_based_position_control_linear(ysw.GRIPPER_ID_R, current_ampere_mag=0.02, linear_pos=desiredGripperPos_mm)
    time.sleep(1)
    gripper.disable(ysw.GRIPPER_ID_R)

************************* [Gripper Connection Starts] **************************

[INFO] SUCCESS: open the port: /dev/rby1_gripper
[INFO] SUCCESS: change the baudrate 2000000

************************* [Gripper Connection is Done] *************************



************************** [Gripper Ping Test Starts] **************************

[INFO] [DXL ID:0] SUCCESS: Dynamixel model number is 1030
[INFO] [DXL ID:1] SUCCESS: Dynamixel model number is 1030

************************* [Gripper Ping Test is Done] **************************



********************* [Gripper [DXL ID:0] setting Starts] **********************

[INFO] SUCCESS: [DXL ID:0] Torque is Disabled
[INFO] SUCCESS: [DXL ID:0] Set Current-based Position Control Mode
[INFO] SUCCESS: [DXL ID:0] Torque is Enabled

********************* [Gripper [DXL ID:0] setting is Done] *********************



********** [Gripper [DXL ID:0] Current-based Position Control Starts] **********

Desired Linear Position = 88.0mm
[INFO] [DXL ID:0]

<span style="
    color:#B8860B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:50px;
    letter-spacing:1px;
    line-height:1.2;
">
Move to post-[Left to Right] posture
</span>

In [20]:
desired_pos = ysw.DEMO_LEFT_TO_RIGHT_POS_AFTER_260723  
ysw.positionCtrl_joint_22dof(robot, desired_pos, minimum_time=10.0, control_hold_time=0.5)

******************** [22-DoF Joint Position Control Starts] ********************


********************* [22-DoF Joint Position Control Ends] *********************





True

<span style="
    color:#B8860B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:50px;
    letter-spacing:1px;
    line-height:1.2;
">
Close Right gripper & Open Left gripper
</span>

In [21]:
object_meanDim = np.mean(leftmost_obj_3Dsize)       # mean value of 3D size of object to grasp
desiredGripperPos = object_meanDim 

# desiredGripperPos_mm = desiredGripperPos * 1000.0 + ysw.GRIPPER_SAFETY_OPEN_OFFSET
desiredGripperPos_mm = desiredGripperPos * 1000.0
if desiredGripperPos_mm >= ysw.GRIPPER_POS_RANGE_LINEAR:
    desiredGripperPos_mm = ysw.GRIPPER_POS_RANGE_LINEAR - 3.0
 
grasp_current = 0.2     # It can be changed

gripper = ysw.Gripper()
gripper.connect(device_name=ysw.GRIPPER_DEVICE_NAME, protocol_version=ysw.DXL_PROTOCOL_VERSION, baudrate=ysw.DXL_BAUDRATE)
gripper.test_ping()

if (moved_arm=="left"):       # Object is in left side of robot
    gripper.initialize_setting(dxl_id=ysw.GRIPPER_ID_R, operation_mode=ysw.DXL_CURRENT_CONTROL_MODE)
    gripper.enable(ysw.GRIPPER_ID_R)
    gripper.write_current_ampere(ysw.GRIPPER_ID_R, grasp_current)
    time.sleep(1)
    gripper.initialize_setting(dxl_id=ysw.GRIPPER_ID_L, operation_mode=ysw.DXL_CURRENT_BASED_POSITION_CONTROL_MODE)
    gripper.current_based_position_control_linear(ysw.GRIPPER_ID_L, current_ampere_mag=0.02, linear_pos=desiredGripperPos_mm)
    time.sleep(1)
    gripper.disable(ysw.GRIPPER_ID_L)

************************* [Gripper Connection Starts] **************************

[INFO] SUCCESS: open the port: /dev/rby1_gripper
[INFO] SUCCESS: change the baudrate 2000000

************************* [Gripper Connection is Done] *************************



************************** [Gripper Ping Test Starts] **************************

[INFO] [DXL ID:0] SUCCESS: Dynamixel model number is 1030
[INFO] [DXL ID:1] SUCCESS: Dynamixel model number is 1030

************************* [Gripper Ping Test is Done] **************************



********************* [Gripper [DXL ID:0] setting Starts] **********************

[INFO] SUCCESS: [DXL ID:0] Torque is Disabled
[INFO] SUCCESS: [DXL ID:0] Set Current Control Mode
[INFO] SUCCESS: [DXL ID:0] Torque is Enabled

********************* [Gripper [DXL ID:0] setting is Done] *********************



********************** [Gripper [DXL ID:0] Enable Starts] **********************

[INFO] SUCCESS: [DXL ID:0] Torque is Enabled

*******************

<span style="
    color:#B8860B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:50px;
    letter-spacing:1px;
    line-height:1.2;
">
Move to pre-[Left to Right] posture
</span>

In [22]:
desired_pos = ysw.DEMO_LEFT_TO_RIGHT_POS_BEFORE_260723  
ysw.positionCtrl_joint_22dof(robot, desired_pos, minimum_time=10.0, control_hold_time=0.5)

******************** [22-DoF Joint Position Control Starts] ********************


********************* [22-DoF Joint Position Control Ends] *********************





True

<span style="
    color:#B8860B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:50px;
    letter-spacing:1px;
    line-height:1.2;
">
Calculate IK for Yellow Box
</span>

In [23]:
# {A}: vision frame
# {B}: "link_head_2" frame
# {S}: "link_torso_5" frame 
# {N}: Goal frame (collecting box) 
# {N1}: Goal frame (collecting box) 
# {N2}: Goal frame (collecting box) 
# {C}: ee_finger_l1/r1 frame
# {D}: final fingertip frame -> same orientation of {C}

manual_offset = 0.05  # [m] for forward x-direction relative to "link_torso_5"

box_pos    = box_obj_avg_sample["center_3d_avg"]
box_3Dsize = box_obj_avg_sample["bbox_3d_size_avg"]

P_BN = ysw.convertPos_vision_to_linkhead2(box_pos)

fk_result = ysw.get_FK(robot, "link_torso_5", "link_head_2")
T_SB = fk_result["T"]
R_SB = fk_result["rotation"]
P_SB = fk_result["position"]

P_BN_extended = np.append(P_BN, 1.0)
P_SN = T_SB @ P_BN_extended
P_SN = P_SN[:3]
          
# Only right hand
moved_arm = "right"
R1 = ysw.rotMat_z(90.0*ysw.DEG2RAD)      # "link_torso_5" frame 기준으로 z축 +90도 회전
R2 = ysw.rotMat_x(-90.0*ysw.DEG2RAD)     # After R1, converted "link_torso_5" frame 기준으로 x축 -90도 회전
R_SC_desired = R1 @ R2              

R_SN = R_SC_desired
T_SN = ysw.make_HT(R_SN, P_SN)    

# "ee_finger_l1/r1" to final end-effector -> should be physically measured
pos_from_eeFinger_to_fingertip = np.array([0, 0, ysw.GRIPPER_Z_OFFSET_FROM_LINK_EE_FINGER]) # P_CD
P_CD = pos_from_eeFinger_to_fingertip
T_CD = ysw.make_HT(np.eye(3), P_CD)    

T_SC_target_for_box = T_SN @ np.linalg.inv(T_CD)
print(f"For Yellow Box: \n{T_SC_target_for_box}\n")
print(f"Moving Arm: {moved_arm}")


*************** [FK of 'link_head_2' from 'link_torso_5' Starts] ***************

[Result]

T:
[[ 0.766   0.      0.6428  0.022 ]
 [-0.      1.      0.     -0.    ]
 [-0.6428  0.      0.766   0.2001]
 [ 0.      0.      0.      1.    ]]


position [m]: [ 0.022  -0.      0.2001]
rpy [rad]: [ 0.      0.6981 -0.    ]
rpy [deg]: [ 0.     39.9997 -0.    ]

**************** [FK of 'link_head_2' from 'link_torso_5' Ends] ****************



For Yellow Box: 
[[ 0.     -0.     -1.      0.5004]
 [ 1.      0.      0.     -0.1934]
 [ 0.     -1.      0.     -0.3077]
 [ 0.      0.      0.      1.    ]]

Moving Arm: right


<span style="
    color:#B8860B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:50px;
    letter-spacing:1px;
    line-height:1.2;
">
Move to Yellow box
</span>

In [24]:
pos_box = T_SC_target_for_box[:3,3]
ori_box = ysw.rotMat_to_rpy(T_SC_target_for_box[:3,:3])       

if (moved_arm == "right"):      # Object is in right side of robot
    cartesian_position_targets_sample = [    
                                            {
                                                "reference_link": "link_torso_5",
                                                "target_link": "ee_finger_r1",
                                                "pos": pos_box + np.array([0.0, 0.0, (np.mean(box_3Dsize)/2.0)]),
                                                "rpy": ori_box                    
                                            },                 
                                        ]

# [rv = "function"] form is Necessary when "get_return=False"
rv = ysw.positionCtrl_cartesian_wholeBody(
    robot, 
    position_targets=cartesian_position_targets_sample, 
    joint_targets=None,
    minimum_time=10,
    control_hold_time=0.5,
    get_return=True
)

*************** [Cartesian Position Control(Whole-Body) Starts] ****************

[INFO] Add (Whole-Body) Cartesian target 0: 'link_torso_5'->'ee_finger_r1', pos=[ 0.5004 -0.1934 -0.0987], rpy=[-1.5708 -0.      1.5708]

**************** [Cartesian Position Control(Whole-Body) Ends] *****************





<span style="
    color:#B8860B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:50px;
    letter-spacing:1px;
    line-height:1.2;
">
Open Gripper to lay
</span>

In [25]:
object_meanDim = np.mean(leftmost_obj_3Dsize)       # mean value of 3D size of object to grasp
desiredGripperPos = object_meanDim 

gripper = ysw.Gripper()
gripper.connect(device_name=ysw.GRIPPER_DEVICE_NAME, protocol_version=ysw.DXL_PROTOCOL_VERSION, baudrate=ysw.DXL_BAUDRATE)
gripper.test_ping()

# desiredGripperPos_mm = desiredGripperPos * 1000.0 + ysw.GRIPPER_SAFETY_OPEN_OFFSET
desiredGripperPos_mm = desiredGripperPos * 1000.0
if desiredGripperPos_mm >= ysw.GRIPPER_POS_RANGE_LINEAR:
    desiredGripperPos_mm = ysw.GRIPPER_POS_RANGE_LINEAR - 3.0

if (moved_arm == "right"):      # Object is in right side of robot
    gripper.initialize_setting(dxl_id=ysw.GRIPPER_ID_R, operation_mode=ysw.DXL_CURRENT_BASED_POSITION_CONTROL_MODE)
    gripper.current_based_position_control_linear(ysw.GRIPPER_ID_R, current_ampere_mag=0.02, linear_pos=desiredGripperPos_mm)
    time.sleep(1)
    gripper.disable(ysw.GRIPPER_ID_R)

************************* [Gripper Connection Starts] **************************

[INFO] SUCCESS: open the port: /dev/rby1_gripper
[INFO] SUCCESS: change the baudrate 2000000

************************* [Gripper Connection is Done] *************************



************************** [Gripper Ping Test Starts] **************************

[INFO] [DXL ID:0] SUCCESS: Dynamixel model number is 1030
[INFO] [DXL ID:1] SUCCESS: Dynamixel model number is 1030

************************* [Gripper Ping Test is Done] **************************



********************* [Gripper [DXL ID:0] setting Starts] **********************

[INFO] SUCCESS: [DXL ID:0] Torque is Disabled
[INFO] SUCCESS: [DXL ID:0] Set Current-based Position Control Mode
[INFO] SUCCESS: [DXL ID:0] Torque is Enabled

********************* [Gripper [DXL ID:0] setting is Done] *********************



********** [Gripper [DXL ID:0] Current-based Position Control Starts] **********

Desired Linear Position = 88.0mm
[INFO] [DXL ID:0]

<span style="
    color:#B8860B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:50px;
    letter-spacing:1px;
    line-height:1.2;
">
Close ALL grippers
</span>

In [34]:
gripper = ysw.Gripper()
gripper.connect(device_name=ysw.GRIPPER_DEVICE_NAME, protocol_version=ysw.DXL_PROTOCOL_VERSION, baudrate=ysw.DXL_BAUDRATE)
gripper.test_ping()

gripper.initialize_setting(dxl_id=ysw.GRIPPER_ID_R, operation_mode=ysw.DXL_CURRENT_BASED_POSITION_CONTROL_MODE)
gripper.initialize_setting(dxl_id=ysw.GRIPPER_ID_L, operation_mode=ysw.DXL_CURRENT_BASED_POSITION_CONTROL_MODE)
gripper.current_based_position_control_linear(ysw.GRIPPER_ID_R, current_ampere_mag=0.02, linear_pos=0.0)
gripper.current_based_position_control_linear(ysw.GRIPPER_ID_L, current_ampere_mag=0.02, linear_pos=0.0)
time.sleep(1)
gripper.disable(ysw.GRIPPER_ID_R)
gripper.disable(ysw.GRIPPER_ID_L)

************************* [Gripper Connection Starts] **************************

[INFO] SUCCESS: open the port: /dev/rby1_gripper
[INFO] SUCCESS: change the baudrate 2000000

************************* [Gripper Connection is Done] *************************



************************** [Gripper Ping Test Starts] **************************

[INFO] [DXL ID:0] SUCCESS: Dynamixel model number is 1030
[INFO] [DXL ID:1] SUCCESS: Dynamixel model number is 1030

************************* [Gripper Ping Test is Done] **************************



********************* [Gripper [DXL ID:0] setting Starts] **********************

[INFO] SUCCESS: [DXL ID:0] Torque is Disabled
[INFO] SUCCESS: [DXL ID:0] Set Current-based Position Control Mode
[INFO] SUCCESS: [DXL ID:0] Torque is Enabled

********************* [Gripper [DXL ID:0] setting is Done] *********************



********************* [Gripper [DXL ID:1] setting Starts] **********************

[INFO] SUCCESS: [DXL ID:1] Torque is Disabled
[INF

True

<span style="color:gray; font-weight:bold; font-size:50px;">
Get Forward Kinematics
</span>

In [17]:
# [Link]:     "link_right_arm_6" 
# -> [Joint]: "tool_right"
# -> [Link]:  "ee_right" 
# -> [Joint]: "gripper_finger_r1"
# -> [Link]:  "ee_finger_r1"    (r1이 ee_right 기준으로 +x 방향, 즉 정면에서 볼때 가까운 방향의 그리퍼임)

# ysw.get_FK(robot, reference_link="base", target_link="link_torso_5")
# ysw.get_FK(robot, reference_link="base", target_link="ee_right")
# ysw.get_FK(robot, reference_link="link_head_2", target_link="ee_right")
# ysw.get_FK(robot, reference_link="link_head_2", target_link="ee_finger_r1")


# End-effector Position Measurement
# fk_result = ysw.get_FK(robot, reference_link="link_head_2", target_link="link_right_arm_1") 
fk_result = ysw.get_FK(robot, reference_link="link_head_2", target_link="link_right_arm_5")    # Actual(pitch joint):             0.65 / FK: 0.652    
fk_result = ysw.get_FK(robot, reference_link="link_head_2", target_link="link_right_arm_6")    # Actual(pitch joint):             0.65 / FK: 0.652 
fk_result = ysw.get_FK(robot, reference_link="link_head_2", target_link="ee_right")            # Actual(gripper prismatic joint): 0.84 / FK: 0.7781 / Not important
fk_result = ysw.get_FK(robot, reference_link="link_head_2", target_link="ee_finger_r1")        # Actual(gripper fingertip):       0.9  / FK: 0.8531 /
# "ee_finger_r1" is located at starting point of prsimatic joint of gripper

************* [FK of 'link_right_arm_5' from 'link_head_2' Starts] *************

[Result]

T:
[[ 1.     -0.006   0.0013 -0.0209]
 [ 0.006   1.     -0.0067 -0.2164]
 [-0.0013  0.0067  1.     -0.6522]
 [ 0.      0.      0.      1.    ]]


position [m]: [-0.0209 -0.2164 -0.6522]
rpy [rad]: [0.0067 0.0013 0.006 ]
rpy [deg]: [0.3856 0.0734 0.342 ]

************** [FK of 'link_right_arm_5' from 'link_head_2' Ends] **************



************* [FK of 'link_right_arm_6' from 'link_head_2' Starts] *************

[Result]

T:
[[ 0.9988  0.0489  0.0013 -0.0209]
 [-0.0488  0.9988 -0.0067 -0.2164]
 [-0.0016  0.0067  1.     -0.6522]
 [ 0.      0.      0.      1.    ]]


position [m]: [-0.0209 -0.2164 -0.6522]
rpy [rad]: [ 0.0067  0.0016 -0.0489]
rpy [deg]: [ 0.381   0.0945 -2.8   ]

************** [FK of 'link_right_arm_6' from 'link_head_2' Ends] **************



***************** [FK of 'ee_right' from 'link_head_2' Starts] *****************

[Result]

T:
[[ 0.9988  0.0489  0.0013 -0.0211]
 [

<span style="color:gray; font-weight:bold; font-size:50px;">
[Test] 3D information of colored object
</span>

In [7]:
# ============================================================
# ROI setting 
# Pixel Origin: (0,0) = 왼쪽 위 꼭짓점
# ============================================================
width = 640
height = 480
pixel_roi_x = None            # None이면 x 전체 사용
pixel_roi_y = [170, height]   # y는 170~480만 사용


# ============================================================
# Start
# ============================================================
d435i = ysw.D435i(rgb_on=True, depth_on=True, pixel=(width,height), fps=30)
d435i.initialize()

d435i.startPipeline()

try:
    while True:
        color_frame, depth_frame = d435i.getRawFrames()

        if color_frame is None or depth_frame is None:
            continue

        color_image, depth_image = d435i.frameToImage(color_frame, depth_frame)
        depth_colorMap = d435i.convertDepthToColorMap(depth_image, alpha=0.03)

        detected_objects, color_image, mask = d435i.detectColorObjectContour3D(
            color_image=color_image,
            depth_frame=depth_frame,
            mask_list=ysw.HSV_MASK_BLACK,       # Can be changed
            minimum_area=1000,                   # Can be changed
            maximum_depth=0.8,                   # ignore distant object larger than x[m] 
            # mask_list=ysw.HSV_MASK_BLACK,       # Can be changed
            # minimum_area=300,                   # Can be changed
            # maximum_depth=0.8,                   # ignore distant object larger than x[m] 
            pixel_roi_x=pixel_roi_x,
            pixel_roi_y=pixel_roi_y,
        )

        for obj in detected_objects:
            print(
                f"[object {obj['id']}] "
                f"center_3d={np.round(obj['center_3d'], 3)}, "
                f"area_2d={np.round(obj['area_2d'], 3)}, "
                f"bbox_3d_size={np.round(obj['bbox_3d_size'], 3)}"
            )
            
        # ============================================================
        # Crop images for imshow (ROI)
        # ============================================================
        color_image_roi = ysw.crop_image_by_roi(
            color_image,
            pixel_roi_x=pixel_roi_x,
            pixel_roi_y=pixel_roi_y
        )

        depth_colorMap_roi = ysw.crop_image_by_roi(
            depth_colorMap,
            pixel_roi_x=pixel_roi_x,
            pixel_roi_y=pixel_roi_y
        )

        mask_roi = ysw.crop_image_by_roi(
            mask,
            pixel_roi_x=pixel_roi_x,
            pixel_roi_y=pixel_roi_y
        )

        # d435i.imshow("color_image", color_image)
        # d435i.imshow("depth_image", depth_colorMap)
        # d435i.imshow("mask", mask)
        d435i.imshow("color_image_roi", color_image_roi)
        d435i.imshow("depth_image_roi", depth_colorMap_roi)
        d435i.imshow("mask_roi", mask_roi)

        key = d435i.waitKey(1)
        if key == 27:
            break

finally:
    d435i.stopPipeline()
    d435i.destroyAllWindows()

[INFO] D435i pipeline started
[object 0] center_3d=[0.138 0.131 0.645], area_2d=2105.5, bbox_3d_size=[0.025 0.127 0.043]
[object 1] center_3d=[0.276 0.126 0.647], area_2d=2456.5, bbox_3d_size=[0.032 0.122 0.037]
[object 0] center_3d=[-0.227  0.154  0.629], area_2d=1721.0, bbox_3d_size=[0.026 0.083 0.039]
[object 1] center_3d=[-0.069  0.154  0.642], area_2d=1572.5, bbox_3d_size=[0.019 0.093 0.049]
[object 2] center_3d=[0.138 0.135 0.647], area_2d=2806.5, bbox_3d_size=[0.026 0.15  0.052]
[object 3] center_3d=[0.267 0.13  0.657], area_2d=5660.0, bbox_3d_size=[0.05  0.15  0.078]
[object 0] center_3d=[-0.228  0.156  0.637], area_2d=1686.0, bbox_3d_size=[0.032 0.08  0.042]
[object 1] center_3d=[-0.069  0.167  0.644], area_2d=1541.5, bbox_3d_size=[0.019 0.092 0.142]
[object 2] center_3d=[0.141 0.141 0.661], area_2d=2817.5, bbox_3d_size=[0.032 0.15  0.091]
[object 3] center_3d=[0.265 0.137 0.659], area_2d=5718.5, bbox_3d_size=[0.12 0.15 0.17]
[object 0] center_3d=[-0.226  0.165  0.63 ], area_2

<span style="color:gray; font-weight:bold; font-size:50px;">
[Test] Move
</span>

In [ ]:
moved_arm = "left"

T_SC_target_before_open_gripper = np.array([
[ 0.  ,    0. ,    -1. ,     0.4518],
 [-1. ,     0. ,    -0. ,     0.0897],
 [ 0.  ,    1. ,     0.  ,   -0.3051],
 [ 0.   ,   0.  ,    0.   ,   1.    ]]
, dtype=float)

pos_before_open_gripper = T_SC_target_before_open_gripper[:3,3]
ori_before_open_gripper = ysw.rotMat_to_rpy(T_SC_target_before_open_gripper[:3,:3])

if (moved_arm == "right"):      # Object is in right side of robot
    cartesian_position_targets_sample = [    
                                            {
                                                "reference_link": "link_torso_5",
                                                "target_link": "ee_finger_r1",
                                                # "pos": pos_before_open_gripper,        
                                                "pos": pos_before_open_gripper + np.array([-0.05, 0.0, 0.05]),            
                                                "rpy": ori_before_open_gripper,                    
                                            },                 
                                        ]
elif (moved_arm == "left"):       # Object is in left side of robot
    cartesian_position_targets_sample = [    
                                            {
                                                "reference_link": "link_torso_5",
                                                "target_link": "ee_finger_l1",
                                                # "pos": pos_before_open_gripper,      
                                                "pos": pos_before_open_gripper + np.array([-0.05, 0.0, 0.05]),                  
                                                "rpy": ori_before_open_gripper,                    
                                            },                 
                                        ]

rv = ysw.positionCtrl_cartesian_wholeBody(
    robot, 
    position_targets=cartesian_position_targets_sample, 
    joint_targets=None,
    minimum_time=10,
    control_hold_time=0.5,
    get_return=True
)

In [ ]:
desired_pos = ysw.DEMO_AFTER_GRASP_POS_260723 
ysw.positionCtrl_joint_22dof(robot, desired_pos, minimum_time=10.0, control_hold_time=0.5)

In [62]:
ysw.getState(robot)

***************************** [Robot State Starts] *****************************

RobotState_A(
  timestamp=datetime.datetime(1970, 1, 1, 9, 12, 11, 62042),
  system_stat=SystemStat(
    cpu_usage=0.041,
    memory_usage=0.0665,
    uptime=3.31e+04,
    program_uptime=3.31e+04
  ),
  battery_state=BatteryState(
    voltage=57,
    current=4,
    level_percent=82
  ),
  power_states=[PowerState(
    state=PowerState.State.PowerOn,
    voltage=0
  ), PowerState(
    state=PowerState.State.PowerOn,
    voltage=0
  ), PowerState(
    state=PowerState.State.PowerOn,
    voltage=0
  ), PowerState(
    state=PowerState.State.PowerOn,
    voltage=57
  )],
  emo_states=[EMOState(
    state=EMOState.State.Released
  )],
  joint_states=[JointState(
    time_since_last_update=datetime.timedelta(microseconds=2),
    is_ready=True,
    fet_state=JointState.FETState.On,
    run_state=JointState.RunState.ControlOn,
    init_state=JointState.InitializationState.Initialized,
    motor_type=1,
    motor_

RobotState_A(
  timestamp=datetime.datetime(1970, 1, 1, 9, 12, 11, 62042),
  system_stat=SystemStat(
    cpu_usage=0.041,
    memory_usage=0.0665,
    uptime=3.31e+04,
    program_uptime=3.31e+04
  ),
  battery_state=BatteryState(
    voltage=57,
    current=4,
    level_percent=82
  ),
  power_states=[PowerState(
    state=PowerState.State.PowerOn,
    voltage=0
  ), PowerState(
    state=PowerState.State.PowerOn,
    voltage=0
  ), PowerState(
    state=PowerState.State.PowerOn,
    voltage=0
  ), PowerState(
    state=PowerState.State.PowerOn,
    voltage=57
  )],
  emo_states=[EMOState(
    state=EMOState.State.Released
  )],
  joint_states=[JointState(
    time_since_last_update=datetime.timedelta(microseconds=2),
    is_ready=True,
    fet_state=JointState.FETState.On,
    run_state=JointState.RunState.ControlOn,
    init_state=JointState.InitializationState.Initialized,
    motor_type=1,
    motor_state=7,
    power_on=True,
    position=0.176,
    velocity=0,
    current=-0.152